**Note (release).** The paper's local-model runs were made with this notebook in the development layout.
Data file names were updated for this release (`data/`); the notebook finds its input files by name and
passes their paths explicitly. Outputs were cleared. Model-loading and generation cells are unchanged.


# BanglaBBQ — open-model evaluation on Kaggle (free GPU)

Same protocol and output format as `evaluation/eval_banglabbq.py` (zero-shot, answer letter A/B/C,
3 cyclic option orders, greedy decoding), so results can be scored together with OpenRouter runs.

**Before running:** Settings → Accelerator **GPU T4 x2**, Internet **On**; input dataset
`banglabbq-eval` attached; for gated models (Llama, Gemma) add a Kaggle secret named `HF_TOKEN`
(Add-ons → Secrets). Change `MODEL_ID` (and `DATASET`) in the next cell, then *Run All*.

`DATASET = "na"` runs the five BanglaBBQ categories (`data/banglabbq_en.jsonl`, `data/banglabbq_bn.jsonl`);
`DATASET = "dt_tm"` runs the balanced sample of the BBQ DT/TM layer
(`evaluation/dt_tm/banglabbq_dt_tm_sample_*.jsonl`, 3,256 records); `DATASET = "vb"` runs the version B
(counter-stereotypical) disambiguated rows (`data/banglabbq_vb_en.jsonl`, `data/banglabbq_vb_bn.jsonl`, 1,864 records).
Each writes to its own results folder.
One 8B model ≈ 20–45 min per language on T4 x2.

In [ ]:
# ---------- settings: change MODEL_ID for each run ----------
MODEL_ID = "Qwen/Qwen3-8B"          # not gated. Others: "meta-llama/Llama-3.1-8B-Instruct" (gated),
                                    # "Qwen/Qwen3-4B", "Qwen/Qwen3-14B" (tight on T4 x2)
DATASET  = "na"         # "na": the 5 BanglaBBQ categories; "dt_tm": BBQ DT/TM sample (evaluation/dt_tm/);
                        # "vb": version B counter-stereotypical disambiguated rows (data/banglabbq_vb_*.jsonl)
LANGS    = ["en", "bn"]
PROMPTS  = [0]          # pilot: [0]; full run: [0, 1, 2]
PERMS    = 3            # cyclic option orders
BATCH    = 16           # lower to 8 if you get CUDA out-of-memory
MAX_NEW  = 8            # answer is one letter
LIMIT    = None         # e.g. 200 for a quick smoke test
START    = 0            # first record to run (split a long run: START=0,LIMIT=1864 then START=1864,LIMIT=None)

In [ ]:
# ---------- locate the uploaded files ----------
import glob, os, shutil, sys
W = "/kaggle/working/repo"
os.makedirs(W + "/output", exist_ok=True); os.makedirs(W + "/evaluation", exist_ok=True)
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found - is the banglabbq-eval dataset attached?")
    return hits[0]
DATA_FILES = {"na":    {"en": "banglabbq_en.jsonl", "bn": "banglabbq_bn.jsonl"},
              "dt_tm": {"en": "banglabbq_dt_tm_sample_eng.jsonl", "bn": "banglabbq_dt_tm_sample_bng.jsonl"},
              "vb":    {"en": "banglabbq_vb_en.jsonl",      "bn": "banglabbq_vb_bn.jsonl"}}[DATASET]
for n in DATA_FILES.values():
    shutil.copy(find(n), f"{W}/output/{n}")
DATA_PATH = {l: f"{W}/output/{n}" for l, n in DATA_FILES.items()}
RESULTS = "/kaggle/working/results" + ("" if DATASET == "na" else "_" + DATASET)
for n in ["eval_banglabbq.py", "score_banglabbq.py", "template_map.csv"]:
    shutil.copy(find(n), f"{W}/evaluation/{n}")
sys.path.insert(0, W + "/evaluation")
import eval_banglabbq as E
print("files ready:", os.listdir(W + "/output"), os.listdir(W + "/evaluation"))
# data fingerprint: which file version this run is reading
from collections import Counter
for _l in LANGS:
    _r = E.load_records(_l, DATA_PATH[_l])
    _unk = Counter(_x[f"ans{j}"] for _x in _r for j in range(3) if _x["answer_info"][f"ans{j}"][1].lower() == "unknown")
    print(f"DATA {DATASET} {_l}: {len(_r)} records, categories {dict(Counter(_x['category'] for _x in _r))}, unknown option {dict(_unk)}")

In [ ]:
!pip install -q -U "transformers>=4.51" accelerate

In [ ]:
# ---------- Hugging Face login (only needed for gated models) ----------
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("logged in to Hugging Face")
except Exception as e:
    print("No HF_TOKEN secret - fine for non-gated models such as Qwen.", type(e).__name__)

In [ ]:
# ---------- load the model ----------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="auto")
model.eval()
print("loaded", MODEL_ID, dtype)

def chat(prompt):
    msgs = [{"role": "user", "content": prompt}]
    # enable_thinking=False turns off Qwen3's reasoning mode; other templates ignore it
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

print(chat(E.build_prompt(E.load_records("en", DATA_PATH["en"])[0], "en", 0, 0)[0]))

In [ ]:
# ---------- run (resumable: re-running skips finished calls) ----------
import json, re, time

def generate(texts):
    """Greedy answers for one batch. On CUDA out-of-memory (the longest Bangla prompts come last),
    free the cache and retry the batch in two halves instead of stopping the run."""
    try:
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        return tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    except torch.cuda.OutOfMemoryError:
        if len(texts) == 1:
            raise
        torch.cuda.empty_cache()
        half = len(texts) // 2
        print(f"  out of GPU memory on a batch of {len(texts)}: retrying as {half} + {len(texts) - half}")
        return generate(texts[:half]) + generate(texts[half:])

def run(lang):
    recs = E.load_records(lang, DATA_PATH[lang])[START:LIMIT]
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", MODEL_ID)
    out = f"{RESULTS}/{safe}__{lang}__p{''.join(map(str, PROMPTS))}_k{PERMS}.jsonl"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    if not os.path.exists(out):     # resume: a partial results file with this name attached as an input
        prev = glob.glob(f"/kaggle/input/**/{os.path.basename(out)}", recursive=True)
        if prev:
            shutil.copy(prev[0], out)
            print("resuming from", prev[0])
    done = set()
    if os.path.exists(out):
        with open(out, encoding="utf-8") as f:
            for l in f:
                try:
                    j = json.loads(l)
                except ValueError:
                    continue                                 # a line cut off by a crash
                done.add((j["example_id"], j["prompt_id"], j["perm"]))
    jobs = []
    for r in recs:
        for p in PROMPTS:
            for k in range(PERMS):
                if (r["example_id"], p, k) not in done:
                    prompt, order = E.build_prompt(r, lang, p, k)
                    jobs.append((r, p, k, order, chat(prompt)))
    jobs.sort(key=lambda j: len(j[4]))                       # similar lengths per batch = faster
    print(f"{lang}: {len(jobs)} calls to run ({len(done)} already done) -> {out}")
    t0, oos = time.time(), 0
    with open(out, "a", encoding="utf-8") as f:
        for i in range(0, len(jobs), BATCH):
            b = jobs[i:i + BATCH]
            texts = generate([j[4] for j in b])
            for (r, p, k, order, _), t in zip(b, texts):
                pred = E.parse_answer(t, r, order)
                oos += pred is None
                f.write(json.dumps({
                    "example_id": r["example_id"], "category": r["category"],
                    "question_index": r["question_index"], "lang": lang, "model": MODEL_ID,
                    "prompt_id": p, "perm": k, "order": order, "seed": 0, "raw": t,
                    "finish": "stop", "error": None, "pred": pred, "usage": None,
                    "provider": "kaggle-transformers", "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
                }, ensure_ascii=False) + "\n")
            if (i // BATCH) % 25 == 0:
                f.flush()
                n = i + len(b)
                print(f"  {n}/{len(jobs)}  out-of-choice {oos}  {n / (time.time() - t0):.1f} calls/s")
    print(f"{lang} done in {(time.time() - t0) / 60:.1f} min, out-of-choice {oos}")
    return out

for lang in LANGS:
    run(lang)

In [ ]:
# ---------- score everything in the results folder of this DATASET ----------
_data = "--data-en " + DATA_PATH["en"] + " --data-bn " + DATA_PATH["bn"]
_report = "/kaggle/working/report" + RESULTS[len("/kaggle/working/results"):]
!python /kaggle/working/repo/evaluation/score_banglabbq.py "{RESULTS}/*.jsonl" {_data} --out {_report} --bootstrap 1000
print(open(f"{_report}/report.md", encoding="utf-8").read())

In [ ]:
# ---------- pack results for download (Output tab -> results_and_report.zip) ----------
import shutil
os.makedirs("/kaggle/working/pack", exist_ok=True)
for d in ["results", "report", "results_dt_tm", "report_dt_tm", "results_vb", "report_vb"]:
    if os.path.exists(f"/kaggle/working/{d}"):
        shutil.copytree(f"/kaggle/working/{d}", f"/kaggle/working/pack/{d}", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/results_and_report", "zip", "/kaggle/working/pack")
print("download /kaggle/working/results_and_report.zip from the Output tab")